### 1. 为什么要部署

把写好的 chain 挂成 HTTP 服务，让其他程序（前端、其他语言、Apifox）都能通过接口调用。核心用两个工具：
- **FastAPI** —— Python 的 Web 框架，负责起 HTTP 服务、定义接口
- **LangServe** —— 把 LangChain 组件（chain）一键转成 REST API

### 2. FastAPI 部署（完整代码，对应 deploy_service.py）

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from langserve import add_routes

# ⭐ 1. 构建要部署的 chain（翻译服务，返回 JSON）
model = init_chat_model(model="deepseek-flash")
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是翻译助手，返回 JSON 格式，包含原文和译文"),
    ("human", "请把以下内容翻译成{language}：{text}"),
])
chain = prompt | model | JsonOutputParser()

# ⭐ 2. 创建 FastAPI 应用（title/version/description 会显示在自动文档里）
app = FastAPI(
    title="基于 LangChain 的服务",
    version="V1.5",
    description="翻译服务",
)

# ⭐ 3. 允许跨域（解决 Apifox/浏览器调用时的 CORS 拦截）
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],       # 学习阶段先放开，生产环境建议指定域名
    allow_credentials=True,
    allow_methods=["*"],       # 允许所有 HTTP 方法
    allow_headers=["*"],       # 允许所有请求头
)

# ⭐ 4. 把 chain 挂到访问路径上（函数和路径一一对应）
add_routes(app, chain, path="/langchainServer")

if __name__ == "__main__":
    import uvicorn  # 服务器
    uvicorn.run(app, host="localhost", port=8000)

#### 逐行要点
- `FastAPI(...)`：创建应用实例，title / version / description 会显示在自动生成的接口文档里
- `CORSMiddleware`：跨域中间件，没有它 Apifox / 浏览器调用会被 CORS 拦截
- `add_routes(app, chain, path=...)`：LangServe 的核心，一行把 chain 变成 REST API
- `uvicorn.run`：启动服务器，监听 8000 端口

启动后浏览器访问 **http://localhost:8000/docs** 可以看到自动生成的接口文档。

### 3. LangServe 客户端调用

服务起来后，有**两种调用方式**：LangChain 客户端（RemoteRunnable）或普通 RESTful API。

In [ ]:
from langserve import RemoteRunnable

# ⭐ 用 LangChain 客户端远程调用（和本地 invoke 写法一样）
client = RemoteRunnable("http://localhost:8000/langchainServer")
print(client.invoke({"language": "意大利文", "text": "我喜欢编程"}))

### 4. 用 RESTful API 调用（其他语言通用）

在 Apifox / Postman 中：
- 方法：POST
- URL：`http://localhost:8000/langchainServer/invoke`
- Body 选 JSON，内容：

```json
{
  "input": {
    "language": "意大利文",
    "text": "为了部落！"
  }
}
```

In [ ]:
# 也可以用 requests 模拟 Apifox 调用，便于在 notebook 里直接测试
import requests

resp = requests.post(
    "http://localhost:8000/langchainServer/invoke",
    json={"input": {"language": "英语", "text": "你好"}},
)
print(resp.json())

### 常见坑
- deploy_service.py 里客户端路径写成了 `lanchainServer`（少了个 g），正确是 `langchainServer`
- 端口被占用：改 `port` 参数（如 8001）
- 调用报 CORS 错：检查是否加了 CORSMiddleware
- 改了 chain 要**重启服务**才生效